# Project 11 — Multi-Agent System

## Overview
We build a **multi-agent research system** where specialised AI agents collaborate to produce high-quality outputs. Each agent has a distinct role and they communicate through a shared blackboard.

**Agent Roles:**
- **Orchestrator**: decomposes tasks and coordinates agents
- **Researcher**: gathers and synthesises information
- **Critic**: reviews and improves output quality
- **Writer**: produces final polished report

## 1. Theory — Multi-Agent Systems

### Why Multiple Agents?
- **Specialisation**: each agent is an expert in its role
- **Parallelism**: agents can work simultaneously on subtasks
- **Quality control**: critic agents catch errors from other agents
- **Scalability**: add more agents to handle more complex tasks

### Communication Architectures

**Blackboard (shared memory)**:
```
Agent A → [Blackboard] ← Agent B
             ↑
           Agent C
```
All agents read/write to a central shared memory. Simple but can cause contention.

**Message passing**:
```
Agent A → Message → Agent B → Message → Agent C
```
Agents communicate via point-to-point messages. More modular.

**Orchestrator-Worker**:
```
Orchestrator → task1 → Worker1
             → task2 → Worker2
             ← result1 ←
             ← result2 ←
```
Central coordinator delegates subtasks to specialist workers.

### Debate-Based Reasoning
Two agents argue opposing sides of a question, then a judge synthesises the best answer. This improves reasoning quality by forcing explicit consideration of counterarguments.

### Challenges
- **Coordination overhead**: inter-agent communication takes time and tokens
- **Error propagation**: mistakes by one agent affect downstream agents
- **Context limits**: shared context grows with each turn
- **Consistency**: agents may produce contradictory outputs

In [ ]:
# pip install anthropic
import json
import time
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any
from datetime import datetime

import anthropic

client = anthropic.Anthropic()
MODEL  = 'claude-haiku-4-5-20251001'

print(f'Anthropic SDK: {anthropic.__version__}')
print(f'Model: {MODEL}')

## 2. Message and Memory Structures

In [ ]:
@dataclass
class Message:
    """A message exchanged between agents."""
    sender:    str
    recipient: str
    content:   str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    msg_type:  str = 'info'  # 'task', 'result', 'feedback', 'info'

    def __str__(self):
        return f'[{self.timestamp[:19]}] {self.sender} → {self.recipient} ({self.msg_type}): {self.content[:80]}'


class AgentMemory:
    """
    Shared blackboard memory for multi-agent collaboration.
    All agents can read/write to this shared state.
    """

    def __init__(self):
        self.messages: List[Message] = []
        self.data:     Dict[str, Any] = {}  # key-value store for shared data
        self.log:      List[str] = []

    def post(self, message: Message):
        """Post a message to the blackboard."""
        self.messages.append(message)
        self.log.append(str(message))

    def get_messages_for(self, agent_name: str) -> List[Message]:
        """Get all messages addressed to a specific agent."""
        return [m for m in self.messages if m.recipient == agent_name]

    def get_results(self) -> List[Message]:
        """Get all result messages."""
        return [m for m in self.messages if m.msg_type == 'result']

    def set(self, key: str, value: Any):
        self.data[key] = value

    def get(self, key: str, default=None) -> Any:
        return self.data.get(key, default)

    def print_log(self):
        print('\n=== Agent Message Log ===')
        for entry in self.log:
            print(f'  {entry}')


print('Message and AgentMemory classes defined.')

## 3. Base Agent Class

In [ ]:
class BaseAgent:
    """Base class for all agents in the system."""

    def __init__(self, name: str, role: str, memory: AgentMemory, model: str = MODEL):
        self.name   = name
        self.role   = role
        self.memory = memory
        self.model  = model
        self.client = anthropic.Anthropic()

    def _call_llm(self, system: str, user: str, max_tokens: int = 512) -> str:
        """Make an LLM API call and return the response text."""
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            system=system,
            messages=[{'role': 'user', 'content': user}]
        )
        return response.content[0].text

    def run(self, task: str, context: str = '') -> str:
        """Override in subclasses."""
        raise NotImplementedError

    def send(self, recipient: str, content: str, msg_type: str = 'result'):
        """Post a message to the shared memory."""
        msg = Message(sender=self.name, recipient=recipient, content=content, msg_type=msg_type)
        self.memory.post(msg)


class ResearcherAgent(BaseAgent):
    """Researches and synthesises information on a subtopic."""

    def run(self, task: str, context: str = '') -> str:
        system = f"""You are an expert researcher. Your role is to provide accurate, 
comprehensive information on topics. Be factual, concise, and well-organised.
Provide specific facts, dates, and technical details where relevant."""

        user = f"""Research the following topic and provide a detailed summary:\n\n{task}

Additional context: {context}

Provide a 150-200 word factual summary with specific details."""

        result = self._call_llm(system, user, max_tokens=400)
        self.send('orchestrator', result)
        return result


class CriticAgent(BaseAgent):
    """Reviews output and provides constructive feedback."""

    def run(self, content: str, context: str = '') -> str:
        system = """You are a critical reviewer. Your job is to identify weaknesses, 
inaccuracies, and areas for improvement in content. Be constructive and specific."""

        user = f"""Review the following content critically:\n\n{content}

Provide:
1. Strengths (2-3 points)
2. Weaknesses or gaps (2-3 points)
3. Specific suggestions for improvement

Be concise (100-150 words)."""

        result = self._call_llm(system, user, max_tokens=300)
        self.send('writer', result)
        return result


class WriterAgent(BaseAgent):
    """Synthesises research into a polished final report."""

    def run(self, research: str, critique: str, topic: str) -> str:
        system = """You are a professional technical writer. Your job is to synthesise 
research into clear, well-structured, engaging content. Address all feedback."""

        user = f"""Write a final report on: {topic}

Based on this research:
{research}

Addressing this critique:
{critique}

Write a polished 250-300 word report with clear sections."""

        result = self._call_llm(system, user, max_tokens=600)
        self.send('orchestrator', result, msg_type='result')
        return result


class OrchestratorAgent(BaseAgent):
    """Coordinates the multi-agent workflow."""

    def decompose(self, task: str) -> List[str]:
        """Break the main task into research subtasks."""
        system = 'You are a task planner. Break complex topics into 3 research subtasks.'
        user   = f"""Break this topic into exactly 3 focused research subtasks:\n{task}

Output ONLY a JSON array of 3 strings, e.g.:
["subtask 1", "subtask 2", "subtask 3"]"""

        result = self._call_llm(system, user, max_tokens=200)
        try:
            # Extract JSON from response
            start = result.find('[')
            end   = result.rfind(']') + 1
            subtasks = json.loads(result[start:end])
            return subtasks[:3]
        except:
            # Fallback subtasks
            return [
                f'History and origins of {task}',
                f'Technical details and mechanisms of {task}',
                f'Applications and impact of {task}'
            ]


print('All agent classes defined.')

## 4. Orchestration Pipeline

In [ ]:
def run_multi_agent_pipeline(topic: str, verbose: bool = True) -> str:
    """
    Full multi-agent research pipeline:
    1. Orchestrator decomposes task into subtasks
    2. Researcher agents work on each subtask
    3. Critic reviews the combined research
    4. Writer produces the final report
    """
    memory = AgentMemory()

    # Create agents
    orchestrator = OrchestratorAgent('orchestrator', 'Task coordinator', memory)
    researcher1  = ResearcherAgent('researcher-1', 'Information gatherer', memory)
    researcher2  = ResearcherAgent('researcher-2', 'Information gatherer', memory)
    researcher3  = ResearcherAgent('researcher-3', 'Information gatherer', memory)
    critic       = CriticAgent('critic', 'Quality reviewer', memory)
    writer       = WriterAgent('writer', 'Report writer', memory)

    researchers = [researcher1, researcher2, researcher3]

    print(f'\n{"="*60}')
    print(f'MULTI-AGENT PIPELINE: {topic}')
    print(f'{"="*60}\n')

    # Step 1: Orchestrator decomposes task
    print('[Orchestrator] Decomposing task into subtasks...')
    subtasks = orchestrator.decompose(topic)
    for i, st in enumerate(subtasks):
        print(f'  Subtask {i+1}: {st}')

    # Step 2: Researchers work on subtasks (sequentially here, could be parallel)
    research_results = []
    for i, (researcher, subtask) in enumerate(zip(researchers, subtasks)):
        print(f'\n[{researcher.name}] Researching: {subtask[:60]}...')
        result = researcher.run(subtask, context=f'Part of a report on: {topic}')
        research_results.append(result)
        print(f'  Done ({len(result)} chars)')

    # Step 3: Critic reviews combined research
    combined_research = '\n\n'.join([
        f'**Subtopic {i+1}: {subtasks[i]}**\n{r}'
        for i, r in enumerate(research_results)
    ])

    print('\n[Critic] Reviewing research...')
    feedback = critic.run(combined_research)
    print(f'  Feedback: {feedback[:100]}...')

    # Step 4: Writer produces final report
    print('\n[Writer] Writing final report...')
    final_report = writer.run(combined_research, feedback, topic)

    # Store results in memory
    memory.set('topic', topic)
    memory.set('subtasks', subtasks)
    memory.set('research', combined_research)
    memory.set('critique', feedback)
    memory.set('final_report', final_report)

    if verbose:
        print('\n' + '='*60)
        print('FINAL REPORT')
        print('='*60)
        print(final_report)
        print('\n' + '='*60)
        memory.print_log()

    return final_report, memory


print('Pipeline function defined.')

## 5. Run the Pipeline

In [ ]:
TOPIC = 'The history and impact of the Transformer architecture in AI'

final_report, memory = run_multi_agent_pipeline(TOPIC, verbose=True)

## 6. Agent Debate Mode
Two agents argue opposing sides, then a judge synthesises the best answer.

In [ ]:
def agent_debate(question: str, n_rounds: int = 2) -> str:
    """
    Structured debate between two agents.
    Agent A argues FOR, Agent B argues AGAINST.
    A judge synthesises the best answer.
    """
    print(f'\n{"="*60}')
    print(f'DEBATE: {question}')
    print(f'{"="*60}')

    memory = AgentMemory()
    debate_history = []

    for round_num in range(1, n_rounds + 1):
        print(f'\n--- Round {round_num} ---')

        # Agent A: argues FOR
        context = '\n'.join(debate_history[-4:]) if debate_history else ''
        pro_agent  = BaseAgent('AgentPRO', 'Proponent', memory)
        pro_system = 'You argue IN FAVOUR of the proposition. Be specific and use evidence.'
        pro_user   = f'Proposition: {question}\n\nPrevious arguments:\n{context}\n\nMake your case (100 words):'
        pro_arg    = pro_agent._call_llm(pro_system, pro_user, max_tokens=200)
        debate_history.append(f'PRO: {pro_arg}')
        print(f'PRO: {pro_arg[:150]}...')

        # Agent B: argues AGAINST
        con_agent  = BaseAgent('AgentCON', 'Opponent', memory)
        con_system = 'You argue AGAINST the proposition. Be specific and address the pro arguments.'
        con_user   = f'Proposition: {question}\n\nArguments so far:\n{"\n".join(debate_history[-4:])}\n\nCounter-argue (100 words):'
        con_arg    = con_agent._call_llm(con_system, con_user, max_tokens=200)
        debate_history.append(f'CON: {con_arg}')
        print(f'CON: {con_arg[:150]}...')

    # Judge synthesises
    judge_agent  = BaseAgent('Judge', 'Synthesis judge', memory)
    judge_system = 'You are a fair judge. Synthesise the best answer from both sides of a debate.'
    judge_user   = f'Question: {question}\n\nDebate transcript:\n{\'\n\'.join(debate_history)}\n\nSynthesize the balanced conclusion (150 words):'
    verdict      = judge_agent._call_llm(judge_system, judge_user, max_tokens=300)

    print(f'\nJUDGE VERDICT:\n{verdict}')
    return verdict


# Run a debate
debate_question = 'Should AI systems be trained on copyrighted data without explicit permission?'
verdict = agent_debate(debate_question, n_rounds=2)

## Summary

| Agent | Role | Input | Output |
|---|---|---|---|
| Orchestrator | Coordinates | High-level task | Subtasks |
| Researcher | Gathers info | Subtask + context | Research summary |
| Critic | Quality control | Combined research | Feedback |
| Writer | Final output | Research + feedback | Polished report |
| Debate agents | Explore tradeoffs | Question | Balanced verdict |

**Key insight**: Agent specialisation improves output quality through division of cognitive labour. The critic-writer loop mirrors how human teams work — draft, review, revise. The debate mode is especially useful for exploring complex questions where the truth lies between extremes.